# Cleaning the data and loading it into silver schema


#### loading all the tables from bronze schema

In [0]:
orders = spark.read.table("ecommerce_data.bronze.orders")
order_items = spark.read.table("ecommerce_data.bronze.order_items")
order_payments = spark.read.table("ecommerce_data.bronze.order_payments")
order_reviews = spark.read.table("ecommerce_data.bronze.order_reviews")
products = spark.read.table("ecommerce_data.bronze.products")
sellers = spark.read.table("ecommerce_data.bronze.sellers")
customers = spark.read.table("ecommerce_data.bronze.customers")
category_translations = spark.read.table("ecommerce_data.bronze.category_translations")
geolocation = spark.read.table("ecommerce_data.bronze.geolocation")


In [0]:
from pyspark.sql.functions import isnull, col, when, count

orders.select([
    count(when(isnull(col(c)), True)).alias(c)
    for c in orders.columns
]).display()

# there are missing values in order_approved_at, order_delivered_carrier_date, and order_delivered_customer_date
# ordered_approved_at - payment approval process failes, order cancelled before approval, system error
# order_delivered_carrier_date - order cancelled before shipping, order never shipped, still being processed, system error
# order_delivered_customer_date - order cancelled before delivery, order never delivered, still being processed, system error

In [0]:
order_items.select([
    count(when(isnull(col(c)), True)).alias(c)
    for c in order_items.columns
]).display()

# order_items has no missing values, all products and sellers are distinct

In [0]:
order_payments.select([
    count(when(isnull(col(c)), True)).alias(c)
    for c in order_payments.columns
]).display()

In [0]:
order_items.select(count("seller_id")).distinct().display()

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# Bronze → Silver Transformation Pipeline
# ─────────────────────────────────────────────────────────────────────────
# This pipeline reads raw Bronze tables (all StringType from Auto Loader),
# applies type conversion, standardization, null handling, deduplication,
# data quality validation, and derived columns, then writes clean Delta
# tables to ecommerce_data.silver.

# --- Imports ---
from pyspark.sql.functions import (
    col, when, count, isnull, isnan, trim, lower, upper, initcap,
    to_timestamp, to_date, year, month, dayofmonth, dayofweek,
    datediff, unix_timestamp, round as spark_round,
    regexp_replace, concat, lit, coalesce, row_number, split, size, expr
)
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType, DecimalType, StringType, TimestampType, DateType
)
from pyspark.sql.window import Window

# --- Create silver schema ---
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_data.silver")
print("Schema ecommerce_data.silver ready.")

In [0]:
# ═══════════════════════════════════════════════════════════════════════════
# 1. ORDERS — Grain: one row per order (PK: order_id)
# ═══════════════════════════════════════════════════════════════════════════
# Source: ecommerce_data.bronze.orders (99,441 rows, all StringType)
#
# Type conversion rationale:
#   - order_id, customer_id: remain string (UUID-like hashes, no numeric meaning)
#   - order_status: remain string (categorical: delivered, shipped, canceled, etc.)
#   - All 4 timestamp columns: string → timestamp (format: "yyyy-MM-dd HH:mm:ss")
#
# Null handling strategy:
#   - order_approved_at (160 nulls): legitimate — order not yet approved or canceled
#   - order_delivered_carrier_date (1,783 nulls): legitimate — order not yet shipped
#   - order_delivered_customer_date (2,965 nulls): legitimate — order not yet delivered
#   - order_id, customer_id, order_status, order_purchase_timestamp: must NOT be null
#
# Validation rules:
#   - order_id must not be null (PK)
#   - customer_id must not be null (FK to customers)
#   - order_status must be one of 8 valid values
#   - order_purchase_timestamp must be parseable
#   - If status = 'delivered', delivered_customer_date should not be null
#
# Derived columns:
#   - approval_duration_hours: time from purchase to approval
#   - delivery_duration_days: time from purchase to customer delivery
#   - carrier_duration_days: time from purchase to carrier handoff
#   - estimated_vs_actual_days: difference between estimated and actual delivery
#   - order_purchase_year, order_purchase_month, order_purchase_day
# ═══════════════════════════════════════════════════════════════════════════

orders_bronze = spark.read.table("ecommerce_data.bronze.orders")

# --- Deduplication: one row per order_id ---
orders_deduped = orders_bronze.dropDuplicates(["order_id"])

# --- Type conversion ---
orders_typed = orders_deduped.select(
    trim(col("order_id")).alias("order_id"),
    trim(col("customer_id")).alias("customer_id"),
    trim(col("order_status")).alias("order_status"),
    to_timestamp(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss").alias("order_purchase_timestamp"),
    to_timestamp(col("order_approved_at"), "yyyy-MM-dd HH:mm:ss").alias("order_approved_at"),
    to_timestamp(col("order_delivered_carrier_date"), "yyyy-MM-dd HH:mm:ss").alias("order_delivered_carrier_date"),
    to_timestamp(col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss").alias("order_delivered_customer_date"),
    to_timestamp(col("order_estimated_delivery_date"), "yyyy-MM-dd HH:mm:ss").alias("order_estimated_delivery_date"),
)

# --- Validation: flag invalid records ---
valid_statuses = ["created", "approved", "processing", "invoiced", "shipped", "delivered", "canceled", "unavailable"]

orders_validated = orders_typed.withColumn(
    "is_valid",
    when(
        isnull(col("order_id")) |
        isnull(col("customer_id")) |
        isnull(col("order_status")) |
        isnull(col("order_purchase_timestamp")) |
        (~col("order_status").isin(valid_statuses)),
        False
    ).otherwise(True)
)

# --- Derived columns ---
orders_silver = orders_validated.withColumn(
    "approval_duration_hours",
    when(col("order_approved_at").isNotNull() & col("order_purchase_timestamp").isNotNull(),
         spark_round(unix_timestamp(col("order_approved_at")) - unix_timestamp(col("order_purchase_timestamp")), 2) / 3600
    ).otherwise(None)
).withColumn(
    "delivery_duration_days",
    when(col("order_delivered_customer_date").isNotNull() & col("order_purchase_timestamp").isNotNull(),
         datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
    ).otherwise(None)
).withColumn(
    "carrier_duration_days",
    when(col("order_delivered_carrier_date").isNotNull() & col("order_purchase_timestamp").isNotNull(),
         datediff(col("order_delivered_carrier_date"), col("order_purchase_timestamp"))
    ).otherwise(None)
).withColumn(
    "estimated_vs_actual_days",
    when(col("order_delivered_customer_date").isNotNull() & col("order_estimated_delivery_date").isNotNull(),
         datediff(col("order_estimated_delivery_date"), col("order_delivered_customer_date"))
    ).otherwise(None)
).withColumn(
    "order_purchase_year", year(col("order_purchase_timestamp"))
).withColumn(
    "order_purchase_month", month(col("order_purchase_timestamp"))
).withColumn(
    "order_purchase_day", dayofmonth(col("order_purchase_timestamp"))
).withColumn(
    "order_purchase_dayofweek", dayofweek(col("order_purchase_timestamp"))
)

# --- Separate valid and invalid records ---
orders_clean = orders_silver.filter(col("is_valid") == True).drop("is_valid")
orders_rejected = orders_silver.filter(col("is_valid") == False)

print(f"Orders total: {orders_silver.count():,}")
print(f"Orders valid: {orders_clean.count():,}")
print(f"Orders rejected: {orders_rejected.count():,}")

In [0]:
# ═════════════════════════════════════════════════════════════════════════
# 2. CUSTOMERS — Grain: one row per customer (PK: customer_id)
# ═══════════════════════════════════════════════════════════════════════════
# Source: ecommerce_data.bronze.customers (99,441 rows, all StringType)
#
# Type conversion rationale:
#   - customer_id, customer_unique_id: remain string (UUID-like hashes)
#   - customer_zip_code_prefix: remain string (5-digit codes, leading zeros matter)
#   - customer_city: remain string, trimmed and lowercased
#   - customer_state: remain string, trimmed and uppercased (2-letter Brazil state codes)
#
# Null handling: No nulls expected in any column. All are required.
#
# Deduplication: one row per customer_id (confirmed no duplicates in bronze)
#
# Validation rules:
#   - customer_id must not be null (PK)
#   - customer_unique_id must not be null (links repeat customers)
#   - customer_zip_code_prefix must be 5 digits
#   - customer_state must be a valid 2-letter Brazilian state code
# ═════════════════════════════════════════════════════════════════════════

customers_bronze = spark.read.table("ecommerce_data.bronze.customers")

# --- Deduplication: one row per customer_id ---
customers_deduped = customers_bronze.dropDuplicates(["customer_id"])

# --- Standardization & type conversion ---
customers_silver = customers_deduped.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_unique_id")).alias("customer_unique_id"),
    trim(col("customer_zip_code_prefix")).alias("customer_zip_code_prefix"),
    lower(trim(col("customer_city"))).alias("customer_city"),
    upper(trim(col("customer_state"))).alias("customer_state"),
)

# --- Validation ---
valid_br_states = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA",
    "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN",
    "RS", "RO", "RR", "SC", "SP", "SE", "TO"
]

customers_validated = customers_silver.withColumn(
    "is_valid",
    when(
        isnull(col("customer_id")) |
        isnull(col("customer_unique_id")) |
        isnull(col("customer_zip_code_prefix")) |
        isnull(col("customer_city")) |
        isnull(col("customer_state")) |
        (~col("customer_state").isin(valid_br_states)),
        False
    ).otherwise(True)
)

customers_clean = customers_validated.filter(col("is_valid") == True).drop("is_valid")
customers_rejected = customers_validated.filter(col("is_valid") == False)

print(f"Customers total: {customers_silver.count():,}")
print(f"Customers valid: {customers_clean.count():,}")
print(f"Customers rejected: {customers_rejected.count():,}")

In [0]:
# ═════════════════════════════════════════════════════════════════════════
# 3. ORDER_ITEMS — Grain: one row per order item (PK: order_id + order_item_id)
# ═════════════════════════════════════════════════════════════════════════
# Source: ecommerce_data.bronze.order_items (112,650 rows, all StringType)
#
# Type conversion rationale:
#   - order_id, product_id, seller_id: remain string (UUID-like hashes)
#   - order_item_id: string -> int (sequential item number within an order, 1-N)
#   - shipping_limit_date: string -> timestamp (seller shipping deadline)
#   - price: string -> decimal(10,2) (monetary value, 2 decimal places)
#   - freight_value: string -> decimal(10,2) (monetary value, 2 decimal places)
#
# Null handling: No nulls expected. All columns required for a valid order item.
#
# Deduplication: one row per (order_id, order_item_id) - confirmed no duplicates
#
# Validation rules:
#   - order_id, product_id, seller_id must not be null
#   - price must be >= 0
#   - freight_value must be >= 0
#   - order_item_id must be >= 1
#
# Derived columns:
#   - total_item_value = price + freight_value
# ═════════════════════════════════════════════════════════════════════════

order_items_bronze = spark.read.table("ecommerce_data.bronze.order_items")

# --- Deduplication: one row per (order_id, order_item_id) ---
order_items_deduped = order_items_bronze.dropDuplicates(["order_id", "order_item_id"])

# --- Type conversion & standardization ---
order_items_silver = order_items_deduped.select(
    trim(col("order_id")).alias("order_id"),
    col("order_item_id").cast(IntegerType()).alias("order_item_id"),
    trim(col("product_id")).alias("product_id"),
    trim(col("seller_id")).alias("seller_id"),
    to_timestamp(col("shipping_limit_date"), "yyyy-MM-dd HH:mm:ss").alias("shipping_limit_date"),
    col("price").cast(DecimalType(10, 2)).alias("price"),
    col("freight_value").cast(DecimalType(10, 2)).alias("freight_value"),
)

# --- Validation ---
order_items_validated = order_items_silver.withColumn(
    "is_valid",
    when(
        isnull(col("order_id")) |
        isnull(col("product_id")) |
        isnull(col("seller_id")) |
        isnull(col("order_item_id")) |
        (col("price") < 0) |
        (col("freight_value") < 0) |
        (col("order_item_id") < 1),
        False
    ).otherwise(True)
)

# --- Derived columns ---
order_items_clean = order_items_validated.filter(col("is_valid") == True).drop("is_valid") \
    .withColumn("total_item_value", spark_round(col("price") + col("freight_value"), 2))

order_items_rejected = order_items_validated.filter(col("is_valid") == False)

print(f"Order items total: {order_items_silver.count():,}")
print(f"Order items valid: {order_items_clean.count():,}")
print(f"Order items rejected: {order_items_rejected.count():,}")

In [0]:
# ═════════════════════════════════════════════════════════════════════════
# 4. ORDER_PAYMENTS — Grain: one row per payment (PK: order_id + payment_sequential)
# ═════════════════════════════════════════════════════════════════════════
# Source: ecommerce_data.bronze.order_payments (103,886 rows, all StringType)
#
# Type conversion rationale:
#   - order_id: remain string (FK to orders)
#   - payment_sequential: string -> int (sequence number for multiple payments)
#   - payment_type: remain string (categorical: credit_card, boleto, voucher, debit_card, not_defined)
#   - payment_installments: string -> int (number of installments)
#   - payment_value: string -> decimal(10,2) (monetary value)
#
# Null handling: No nulls expected. All columns required.
#
# Deduplication: one row per (order_id, payment_sequential) - confirmed no duplicates
#
# Validation rules:
#   - order_id must not be null
#   - payment_value must be >= 0
#   - payment_installments must be >= 0
#   - payment_type must be one of 5 valid values
#   - payment_sequential must be >= 1
# ═════════════════════════════════════════════════════════════════════════

order_payments_bronze = spark.read.table("ecommerce_data.bronze.order_payments")

# --- Deduplication: one row per (order_id, payment_sequential) ---
order_payments_deduped = order_payments_bronze.dropDuplicates(["order_id", "payment_sequential"])

# --- Type conversion & standardization ---
order_payments_silver = order_payments_deduped.select(
    trim(col("order_id")).alias("order_id"),
    col("payment_sequential").cast(IntegerType()).alias("payment_sequential"),
    lower(trim(col("payment_type"))).alias("payment_type"),
    col("payment_installments").cast(IntegerType()).alias("payment_installments"),
    col("payment_value").cast(DecimalType(10, 2)).alias("payment_value"),
)

# --- Validation ---
valid_payment_types = ["credit_card", "boleto", "voucher", "debit_card", "not_defined"]

order_payments_validated = order_payments_silver.withColumn(
    "is_valid",
    when(
        isnull(col("order_id")) |
        isnull(col("payment_sequential")) |
        isnull(col("payment_type")) |
        isnull(col("payment_installments")) |
        isnull(col("payment_value")) |
        (col("payment_value") < 0) |
        (col("payment_installments") < 0) |
        (col("payment_sequential") < 1) |
        (~col("payment_type").isin(valid_payment_types)),
        False
    ).otherwise(True)
)

order_payments_clean = order_payments_validated.filter(col("is_valid") == True).drop("is_valid")
order_payments_rejected = order_payments_validated.filter(col("is_valid") == False)

print(f"Order payments total: {order_payments_silver.count():,}")
print(f"Order payments valid: {order_payments_clean.count():,}")
print(f"Order payments rejected: {order_payments_rejected.count():,}")

In [0]:
# ═════════════════════════════════════════════════════════════════════════
# 5. ORDER_REVIEWS — Grain: one row per review (PK: review_id)
# ═════════════════════════════════════════════════════════════════════════
# Source: ecommerce_data.bronze.order_reviews (104,162 rows, all StringType)
#
# DATA QUALITY ISSUE: review_score contains corrupted values — Auto Loader
# rescue mode mixed review comment text into review_score for some rows.
# The valid range is integers 1-5.
#
# Type conversion rationale:
#   - review_id, order_id: remain string (UUID-like hashes)
#   - review_score: string -> int (should be 1-5, garbage text becomes null on cast)
#   - review_comment_title, review_comment_message: remain string, trimmed
#   - review_creation_date: string -> timestamp
#   - review_answer_timestamp: string -> timestamp
#
# Null handling:
#   - review_comment_title (92,157 nulls): legitimate — most reviews have no title
#   - review_comment_message (63,079 nulls): legitimate — most reviews have no comment
#   - review_id (1 null), order_id (2,236 nulls), review_score (2,380 nulls): invalid
#
# Deduplication: one row per review_id - confirmed no duplicates
#
# Validation rules:
#   - review_id must not be null (PK)
#   - review_score must be an integer between 1 and 5 (corrupted values flagged)
#   - order_id should not be null (FK to orders)
#
# Derived columns:
#   - review_response_hours: time from review creation to answer
# ═════════════════════════════════════════════════════════════════════════

order_reviews_bronze = spark.read.table("ecommerce_data.bronze.order_reviews")

# --- Deduplication: one row per review_id ---
order_reviews_deduped = order_reviews_bronze.dropDuplicates(["review_id"])

# --- Type conversion & standardization ---
# Attempt to cast review_score to int; non-numeric values become null
order_reviews_typed = order_reviews_deduped.select(
    trim(col("review_id")).alias("review_id"),
    trim(col("order_id")).alias("order_id"),
    expr("try_cast(review_score as int)").alias("review_score"),
    trim(col("review_comment_title")).alias("review_comment_title"),
    trim(col("review_comment_message")).alias("review_comment_message"),
    expr("try_to_timestamp(review_creation_date, 'yyyy-MM-dd HH:mm:ss')").alias("review_creation_date"),
    expr("try_to_timestamp(review_answer_timestamp, 'yyyy-MM-dd HH:mm:ss')").alias("review_answer_timestamp")
)

# --- Validation: review_score must be 1-5 ---
order_reviews_validated = order_reviews_typed.withColumn(
    "is_valid",
    when(
        isnull(col("review_id")) |
        isnull(col("review_score")) |
        (col("review_score") < 1) |
        (col("review_score") > 5),
        False
    ).otherwise(True)
)

# --- Derived columns ---
order_reviews_clean = order_reviews_validated.filter(col("is_valid") == True).drop("is_valid") \
    .withColumn(
        "review_response_hours",
        when(col("review_answer_timestamp").isNotNull() & col("review_creation_date").isNotNull(),
             spark_round(unix_timestamp(col("review_answer_timestamp")) - unix_timestamp(col("review_creation_date")), 2) / 3600
        ).otherwise(None)
    )

order_reviews_rejected = order_reviews_validated.filter(col("is_valid") == False)

print(f"Order reviews total: {order_reviews_typed.count():,}")
print(f"Order reviews valid: {order_reviews_clean.count():,}")
print(f"Order reviews rejected (incl. corrupted scores): {order_reviews_rejected.count():,}")

In [0]:
# ═════════════════════════════════════════════════════════════════════════
# 6. PRODUCTS — Grain: one row per product (PK: product_id)
# ═════════════════════════════════════════════════════════════════════════
# Source: ecommerce_data.bronze.products (32,951 rows, all StringType)
#
# Type conversion rationale:
#   - product_id: remain string (UUID-like hash)
#   - product_category_name: remain string (Portuguese category name)
#   - product_name_lenght -> product_name_length: string -> int (char count)
#   - product_description_lenght -> product_description_length: string -> int
#   - product_photos_qty: string -> int (count of photos)
#   - product_weight_g: string -> int (weight in grams, whole number)
#   - product_length_cm, product_height_cm, product_width_cm: string -> decimal(10,2)
#
# Column name fix: "lenght" -> "length" (typo in original Olist dataset)
#
# Null handling:
#   - product_category_name (610 nulls): legitimate - some products uncategorized
#   - product_name_length, description_length, photos_qty (610 each): tied to same records
#   - product_weight_g, dimensions (2 nulls each): invalid - physical product must have dimensions
#
# Deduplication: one row per product_id - confirmed no duplicates
#
# Validation rules:
#   - product_id must not be null (PK)
#   - product_weight_g, dimensions must be >= 0 if not null
#   - product_photos_qty must be >= 0 if not null
# ═════════════════════════════════════════════════════════════════════════

products_bronze = spark.read.table("ecommerce_data.bronze.products")

# --- Deduplication: one row per product_id ---
products_deduped = products_bronze.dropDuplicates(["product_id"])

# --- Type conversion, standardization & column name fix ---
products_silver = products_deduped.select(
    trim(col("product_id")).alias("product_id"),
    lower(trim(col("product_category_name"))).alias("product_category_name"),
    col("product_name_lenght").cast(IntegerType()).alias("product_name_length"),
    col("product_description_lenght").cast(IntegerType()).alias("product_description_length"),
    col("product_photos_qty").cast(IntegerType()).alias("product_photos_qty"),
    col("product_weight_g").cast(IntegerType()).alias("product_weight_g"),
    col("product_length_cm").cast(DecimalType(10, 2)).alias("product_length_cm"),
    col("product_height_cm").cast(DecimalType(10, 2)).alias("product_height_cm"),
    col("product_width_cm").cast(DecimalType(10, 2)).alias("product_width_cm"),
)

# --- Validation ---
products_validated = products_silver.withColumn(
    "is_valid",
    when(
        isnull(col("product_id")) |
        (col("product_weight_g") < 0) |
        (col("product_length_cm") < 0) |
        (col("product_height_cm") < 0) |
        (col("product_width_cm") < 0) |
        (col("product_photos_qty") < 0),
        False
    ).otherwise(True)
)

products_clean = products_validated.filter(col("is_valid") == True).drop("is_valid")
products_rejected = products_validated.filter(col("is_valid") == False)

print(f"Products total: {products_silver.count():,}")
print(f"Products valid: {products_clean.count():,}")
print(f"Products rejected: {products_rejected.count():,}")

In [0]:
# ═════════════════════════════════════════════════════════════════════════
# 7. SELLERS — Grain: one row per seller (PK: seller_id)
# ═════════════════════════════════════════════════════════════════════════
# Source: ecommerce_data.bronze.sellers (3,095 rows, all StringType)
#
# Type conversion rationale:
#   - seller_id: remain string (UUID-like hash)
#   - seller_zip_code_prefix: remain string (5-digit codes, leading zeros matter)
#   - seller_city: remain string, trimmed and lowercased
#   - seller_state: remain string, trimmed and uppercased (2-letter Brazil state codes)
#
# Null handling: No nulls in any column. All are required.
#
# Deduplication: one row per seller_id - confirmed no duplicates
#
# Validation rules:
#   - seller_id must not be null (PK)
#   - seller_state must be a valid 2-letter Brazilian state code
# ═════════════════════════════════════════════════════════════════════════

sellers_bronze = spark.read.table("ecommerce_data.bronze.sellers")

# --- Deduplication: one row per seller_id ---
sellers_deduped = sellers_bronze.dropDuplicates(["seller_id"])

# --- Standardization & type conversion ---
sellers_silver = sellers_deduped.select(
    trim(col("seller_id")).alias("seller_id"),
    trim(col("seller_zip_code_prefix")).alias("seller_zip_code_prefix"),
    lower(trim(col("seller_city"))).alias("seller_city"),
    upper(trim(col("seller_state"))).alias("seller_state"),
)

# --- Validation ---
sellers_validated = sellers_silver.withColumn(
    "is_valid",
    when(
        isnull(col("seller_id")) |
        isnull(col("seller_zip_code_prefix")) |
        isnull(col("seller_city")) |
        isnull(col("seller_state")) |
        (~col("seller_state").isin(valid_br_states)),
        False
    ).otherwise(True)
)

sellers_clean = sellers_validated.filter(col("is_valid") == True).drop("is_valid")
sellers_rejected = sellers_validated.filter(col("is_valid") == False)

print(f"Sellers total: {sellers_silver.count():,}")
print(f"Sellers valid: {sellers_clean.count():,}")
print(f"Sellers rejected: {sellers_rejected.count():,}")

In [0]:
# ═════════════════════════════════════════════════════════════════════════
# 8. GEOLOCATION — Grain: one row per zip code + lat/lng combination
# ═════════════════════════════════════════════════════════════════════════
# Source: ecommerce_data.bronze.geolocation (1,000,163 rows, all StringType)
#
# NOTE: This table has multiple rows per zip_code_prefix — each represents a
# different coordinate within the same postal area. This is LEGITIMATE and
# should NOT be deduplicated by zip code alone.
#
# Type conversion rationale:
#   - geolocation_zip_code_prefix: remain string (5-digit codes, leading zeros matter)
#   - geolocation_lat: string -> double (decimal degrees latitude)
#   - geolocation_lng: string -> double (decimal degrees longitude)
#   - geolocation_city: remain string, trimmed and lowercased
#   - geolocation_state: remain string, trimmed and uppercased
#
# Null handling: No nulls in any column. All are required.
#
# Deduplication: Drop exact duplicates (all columns match) — legitimate
# multiple coordinates per zip code are preserved.
#
# Validation rules:
#   - geolocation_lat must be between -90 and 90 (valid latitude range)
#   - geolocation_lng must be between -180 and 180 (valid longitude range)
#   - geolocation_zip_code_prefix must not be null
#   - geolocation_state must be a valid 2-letter Brazilian state code
# ═════════════════════════════════════════════════════════════════════════

geolocation_bronze = spark.read.table("ecommerce_data.bronze.geolocation")

# --- Deduplication: remove only exact duplicate rows ---
geolocation_deduped = geolocation_bronze.dropDuplicates()

# --- Type conversion & standardization ---
geolocation_silver = geolocation_deduped.select(
    trim(col("geolocation_zip_code_prefix")).alias("geolocation_zip_code_prefix"),
    col("geolocation_lat").cast(DoubleType()).alias("geolocation_lat"),
    col("geolocation_lng").cast(DoubleType()).alias("geolocation_lng"),
    lower(trim(col("geolocation_city"))).alias("geolocation_city"),
    upper(trim(col("geolocation_state"))).alias("geolocation_state"),
)

# --- Validation ---
geolocation_validated = geolocation_silver.withColumn(
    "is_valid",
    when(
        isnull(col("geolocation_zip_code_prefix")) |
        isnull(col("geolocation_lat")) |
        isnull(col("geolocation_lng")) |
        isnull(col("geolocation_city")) |
        isnull(col("geolocation_state")) |
        (col("geolocation_lat") < -90) | (col("geolocation_lat") > 90) |
        (col("geolocation_lng") < -180) | (col("geolocation_lng") > 180) |
        (~col("geolocation_state").isin(valid_br_states)),
        False
    ).otherwise(True)
)

geolocation_clean = geolocation_validated.filter(col("is_valid") == True).drop("is_valid")
geolocation_rejected = geolocation_validated.filter(col("is_valid") == False)

print(f"Geolocation total: {geolocation_silver.count():,}")
print(f"Geolocation valid: {geolocation_clean.count():,}")
print(f"Geolocation rejected: {geolocation_rejected.count():,}")

In [0]:
# ═════════════════════════════════════════════════════════════════════════
# 9. CATEGORY_TRANSLATION — Grain: one row per category (PK: product_category_name)
# ═════════════════════════════════════════════════════════════════════════
# Source: ecommerce_data.bronze.category_translations (71 rows, all StringType)
#
# Type conversion rationale:
#   - product_category_name: remain string (Portuguese category name)
#   - product_category_name_english: remain string (English translation)
#   Both are categorical text, no numeric or date fields.
#
# Null handling: No nulls in either column. Both are required.
#
# Deduplication: one row per product_category_name - confirmed no duplicates
#
# Validation rules:
#   - product_category_name must not be null (PK)
#   - product_category_name_english must not be null
# ═════════════════════════════════════════════════════════════════════════

category_translations_bronze = spark.read.table("ecommerce_data.bronze.category_translations")

# --- Deduplication: one row per product_category_name ---
category_translations_deduped = category_translations_bronze.dropDuplicates(["product_category_name"])

# --- Standardization ---
category_translations_silver = category_translations_deduped.select(
    lower(trim(col("product_category_name"))).alias("product_category_name"),
    lower(trim(col("product_category_name_english"))).alias("product_category_name_english"),
)

# --- Validation ---
category_translations_validated = category_translations_silver.withColumn(
    "is_valid",
    when(
        isnull(col("product_category_name")) |
        isnull(col("product_category_name_english")),
        False
    ).otherwise(True)
)

category_translations_clean = category_translations_validated.filter(col("is_valid") == True).drop("is_valid")
category_translations_rejected = category_translations_validated.filter(col("is_valid") == False)

print(f"Category translations total: {category_translations_silver.count():,}")
print(f"Category translations valid: {category_translations_clean.count():,}")
print(f"Category translations rejected: {category_translations_rejected.count():,}")

In [0]:
# ═════════════════════════════════════════════════════════════════════════
# WRITE ALL SILVER TABLES USING MERGE (INCREMENTAL UPSERT)
# ═════════════════════════════════════════════════════════════════════════
# Uses MERGE INTO for incremental upserts instead of full overwrite:
#   - New records:      INSERT  (WHEN NOT MATCHED)
#   - Updated records:  UPDATE  (WHEN MATCHED)
#   - Late-arriving:    MERGE handles naturally — no reprocessing of full history
#   - Duplicates:       already deduped in transformation cells via dropDuplicates
#
# First run:   tables created via overwrite (if they don't exist yet)
# Later runs:  MERGE upserts only new/changed data by primary key
# ═════════════════════════════════════════════════════════════════════════

# (table_name, DataFrame, primary_key_columns)
silver_tables = {
    "orders":             (orders_clean,              ["order_id"]),
    "customers":          (customers_clean,           ["customer_id"]),
    "order_items":         (order_items_clean,          ["order_id", "order_item_id"]),
    "order_payments":      (order_payments_clean,       ["order_id", "payment_sequential"]),
    "order_reviews":       (order_reviews_clean,        ["review_id"]),
    "products":            (products_clean,            ["product_id"]),
    "sellers":             (sellers_clean,             ["seller_id"]),
    "geolocation":         (geolocation_clean,         ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"]),
    "category_translation": (category_translations_clean, ["product_category_name"]),
}

for table_name, (df, key_cols) in silver_tables.items():
    full_name = f"ecommerce_data.silver.{table_name}"
    temp_view = f"silver_{table_name}_updates"

    # Deduplicate on primary key to prevent MERGE ambiguity (multiple source rows matching same target row)
    df_deduped = df.dropDuplicates(key_cols)

    # Register deduplicated DataFrame as temp view for MERGE
    df_deduped.createOrReplaceTempView(temp_view)

    # Build merge condition on primary key columns
    merge_on = " AND ".join([f"target.{c} = source.{c}" for c in key_cols])

    if spark.catalog.tableExists(full_name):
        # ── Incremental MERGE: upsert new and changed records ──
        spark.sql(f"""
            MERGE INTO {full_name} AS target
            USING {temp_view} AS source
            ON {merge_on}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        row_count = spark.read.table(full_name).count()
        print(f"  {table_name:25s}  MERGED on {key_cols}  ->  {row_count:,} rows total")
    else:
        # ── First run: create table ──
        df.write.format("delta").mode("overwrite").saveAsTable(full_name)
        print(f"  {table_name:25s}  CREATED with {df.count():,} rows")

print("\nAll silver tables written successfully.")
print("\nRejected records summary:")
print(f"  orders rejected:            {orders_rejected.count():,}")
print(f"  customers rejected:         {customers_rejected.count():,}")
print(f"  order_items rejected:       {order_items_rejected.count():,}")
print(f"  order_payments rejected:    {order_payments_rejected.count():,}")
print(f"  order_reviews rejected:     {order_reviews_rejected.count():,}")
print(f"  products rejected:         {products_rejected.count():,}")
print(f"  sellers rejected:           {sellers_rejected.count():,}")
print(f"  geolocation rejected:       {geolocation_rejected.count():,}")
print(f"  category_translations rej:  {category_translations_rejected.count():,}")

In [0]:
# ═════════════════════════════════════════════════════════════════════════
# VALIDATION QUERIES — Verify Silver table integrity
# ═════════════════════════════════════════════════════════════════════════
# Runs a suite of data-quality checks on the written Silver tables.
# Each check returns 0 if passing, or a count of violating rows.
# ═════════════════════════════════════════════════════════════════════════

checks = [
    ("orders: no null order_id",          "SELECT count(*) FROM ecommerce_data.silver.orders WHERE order_id IS NULL"),
    ("orders: no null customer_id",      "SELECT count(*) FROM ecommerce_data.silver.orders WHERE customer_id IS NULL"),
    ("orders: valid order_status",      "SELECT count(*) FROM ecommerce_data.silver.orders WHERE order_status NOT IN ('created','approved','processing','invoiced','shipped','delivered','canceled','unavailable')"),
    ("customers: no null customer_id",  "SELECT count(*) FROM ecommerce_data.silver.customers WHERE customer_id IS NULL"),
    ("customers: valid state code",     "SELECT count(*) FROM ecommerce_data.silver.customers WHERE customer_state NOT IN ('AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG','PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO')"),
    ("order_items: no null order_id",    "SELECT count(*) FROM ecommerce_data.silver.order_items WHERE order_id IS NULL"),
    ("order_items: price >= 0",         "SELECT count(*) FROM ecommerce_data.silver.order_items WHERE price < 0"),
    ("order_items: freight >= 0",       "SELECT count(*) FROM ecommerce_data.silver.order_items WHERE freight_value < 0"),
    ("order_payments: value >= 0",      "SELECT count(*) FROM ecommerce_data.silver.order_payments WHERE payment_value < 0"),
    ("order_payments: installments >= 0","SELECT count(*) FROM ecommerce_data.silver.order_payments WHERE payment_installments < 0"),
    ("order_reviews: score 1-5",        "SELECT count(*) FROM ecommerce_data.silver.order_reviews WHERE review_score < 1 OR review_score > 5"),
    ("order_reviews: no null review_id","SELECT count(*) FROM ecommerce_data.silver.order_reviews WHERE review_id IS NULL"),
    ("products: no null product_id",    "SELECT count(*) FROM ecommerce_data.silver.products WHERE product_id IS NULL"),
    ("products: weight >= 0",           "SELECT count(*) FROM ecommerce_data.silver.products WHERE product_weight_g < 0"),
    ("sellers: no null seller_id",      "SELECT count(*) FROM ecommerce_data.silver.sellers WHERE seller_id IS NULL"),
    ("geolocation: valid lat range",    "SELECT count(*) FROM ecommerce_data.silver.geolocation WHERE geolocation_lat < -90 OR geolocation_lat > 90"),
    ("geolocation: valid lng range",    "SELECT count(*) FROM ecommerce_data.silver.geolocation WHERE geolocation_lng < -180 OR geolocation_lng > 180"),
    ("category_translation: no nulls",  "SELECT count(*) FROM ecommerce_data.silver.category_translation WHERE product_category_name IS NULL OR product_category_name_english IS NULL"),
]

all_pass = True
for label, sql in checks:
    result = spark.sql(sql).collect()[0][0]
    status = "PASS" if result == 0 else "FAIL"
    if result != 0:
        all_pass = False
    print(f"  [{status}] {label}: {result} violations")

# Row count summary
print("\nSilver table row counts:")
for t in ["orders","customers","order_items","order_payments","order_reviews","products","sellers","geolocation","category_translation"]:
    cnt = spark.sql(f"SELECT count(*) FROM ecommerce_data.silver.{t}").collect()[0][0]
    print(f"  {t:25s}  {cnt:,} rows")

print(f"\n{'='*60}")
print(f"Overall: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")
print(f"{'='*60}")

# Bronze → Silver Architecture & Design Decisions

## 1. Architecture Overview

The Bronze → Silver pipeline follows a **medallion architecture** pattern:

- **Bronze** (`ecommerce_data.bronze`): Raw data ingested via Auto Loader from JSON files. All columns are `StringType` because Auto Loader inferred the schema with `rescue` mode. Bronze tables remain raw and are never modified — they serve as the immutable source of truth for reprocessing.
- **Silver** (`ecommerce_data.silver`): Cleaned, typed, standardized, validated, and deduplicated tables. Silver is the **trusted data layer** where analysts and downstream pipelines can rely on correct data types, no duplicates, and enforced data-quality rules.

The pipeline is idempotent: re-running it produces the same Silver tables without side effects.

## 2. Bronze Table Profiles & Grain

| Table | Rows | Grain (PK) | Key Columns |
|---|---|---|---|
| `orders` | 99,441 | One row per order (`order_id`) | order_status, 4 timestamp columns |
| `customers` | 99,441 | One row per customer (`customer_id`) | customer_unique_id, city, state, zip |
| `order_items` | 112,650 | One row per order item (`order_id` + `order_item_id`) | product_id, seller_id, price, freight_value |
| `order_payments` | 103,886 | One row per payment (`order_id` + `payment_sequential`) | payment_type, installments, payment_value |
| `order_reviews` | 104,162 | One row per review (`review_id`) | order_id, review_score, comment_title/message |
| `products` | 32,951 | One row per product (`product_id`) | category_name, dimensions, weight |
| `sellers` | 3,095 | One row per seller (`seller_id`) | city, state, zip |
| `geolocation` | 1,000,163 | One row per zip+lat+lng combo (no single PK) | zip_code_prefix, lat, lng, city, state |
| `category_translations` | 71 | One row per category (`product_category_name`) | product_category_name_english |

## 3. Data Quality Issues Handled

| Table | Issue | Silver Handling |
|---|---|---|
| All | All columns `StringType` from Auto Loader | Cast to appropriate types (timestamp, int, decimal, double) |
| All | `_rescued_data` column from Auto Loader rescue mode | Dropped (not selected into Silver) |
| `order_reviews` | `review_score` corrupted with Portuguese comment text | `try_cast(review_score as int)` — garbage becomes null, then rejected |
| `order_reviews` | Timestamp columns also corrupted | `try_to_timestamp()` — unparseable values become null |
| `products` | Column names have typo: `lenght` → `length` | Renamed in Silver select aliases |
| `geolocation` | Multiple rows per zip code | Intentionally preserved — `dropDuplicates()` only removes exact duplicates |

## 4. Joins: Silver vs Gold

### Silver Layer: No Joins

Silver tables are kept **separate and independent**. This is intentional:
- Each table represents a clean entity at its natural grain.
- Keeping them separate preserves reusability — different Gold models need different join combinations.
- Joining at Silver would denormalize the data prematurely and create wide, hard-to-maintain tables.

### Gold Layer: Where Joins Belong

Joins should be performed in the Gold layer for dimensional modeling and analytics:

| Gold Use Case | Tables to Join | Join Keys |
|---|---|---|
| Order analysis with customer info | `orders` ⨝ `customers` | `orders.customer_id = customers.customer_id` |
| Revenue by product/category | `order_items` ⨝ `products` ⨝ `category_translation` | `order_items.product_id = products.product_id`, `products.product_category_name = category_translation.product_category_name` |
| Seller performance | `order_items` ⨝ `sellers` | `order_items.seller_id = sellers.seller_id` |
| Payment analysis | `orders` ⨝ `order_payments` | `orders.order_id = order_payments.order_id` |
| Review analysis | `orders` ⨝ `order_reviews` | `orders.order_id = order_reviews.order_id` |
| Geographic analysis | `customers`/`sellers` ⨝ `geolocation` | `zip_code_prefix` join (many-to-many — aggregate geolocation first) |

## 5. Incremental Processing Design

### Current Implementation: MERGE (Upsert)

The write cell uses `MERGE INTO` instead of `mode("overwrite")`:

- **New records**: `WHEN NOT MATCHED THEN INSERT *` — inserts rows that don't exist in Silver yet
- **Updated records**: `WHEN MATCHED THEN UPDATE SET *` — updates existing rows when Bronze data changes (e.g., an order status changes from `shipped` to `delivered`)
- **Duplicate records**: Handled before MERGE via `dropDuplicates()` in the transformation cells
- **Late-arriving records**: MERGE naturally handles these — a record arriving late simply matches on PK and updates the existing row

### How It Works Run-by-Run

1. **First run**: Tables don't exist → created via `saveAsTable` with `overwrite` mode
2. **Subsequent runs**: Tables exist → `MERGE INTO` upserts only new/changed data by primary key
3. **Reprocessing**: If Bronze schema changes or a full reload is needed, drop the Silver table and re-run (the pipeline detects the missing table and recreates it)

### Why Not Overwrite?

`mode("overwrite")` recreates the entire table every run, which:
- Loses Delta table history and time-travel capability
- Is inefficient for large tables (re-writes all data even if only a few rows changed)
- Doesn't scale to incremental Bronze ingestion from Auto Loader

## 6. Conversion to Lakeflow Spark Declarative Pipeline (SDP)

This notebook can be converted to a **Lakeflow Spark Declarative Pipeline** for automated, incremental processing. The key concept is replacing manual `read.table` + `write` with **streaming tables** and **materialized views** declared using the `databricks.pipelines` API.

### Mapping from Notebook to SDP

| Notebook Concept | SDP Equivalent |
|---|---|
| `spark.read.table("bronze.orders")` | `dp.read_stream("ecommerce_data.bronze.orders")` or `dp.read("...")` for batch |
| `dropDuplicates()`, `select()`, `withColumn()` | Same PySpark transformations, wrapped in `@dp.table` or `@dp.streaming_table` |
| `MERGE INTO` write cell | `apply_changes()` for CDC-style upserts, or let SDP handle dedup via `expect` rules |
| Validation rules | `@dp.expect` and `@dp.expect_all` data quality constraints |
| Rejected records | `on_violation = "fail"` or `on_violation = "drop"` in expectations |

### What SDP Adds

- **Automatic incremental processing**: Streaming tables process only new Bronze data each run
- **Built-in checkpointing**: No need for manual checkpoint management
- **Schema evolution**: `schema_evolution_mode = "rescue"` handled at the SDP level
- **Data quality expectations**: `@dp.expect` annotations replace manual `is_valid` column logic
- **Automatic retries and error handling**: Pipeline-level orchestration

### When to Convert

- **Now**: The notebook pipeline is production-quality for batch processing. If Bronze is loaded in batches (not streaming), the current MERGE approach is sufficient.
- **Later**: Convert to SDP when Bronze ingestion becomes continuous (Auto Loader streaming), when you need automated scheduling, or when multiple pipelines share Bronze tables and need coordinated refresh.

### Recommended SDP Structure

```
bronze/  (Auto Loader → raw Delta tables — already done)
silver/  (SDP streaming tables with expectations — future)
gold/    (SDP materialized views with joins — future)
```

## 7. Implementation: Now vs Later

| Component | Status | Notes |
|---|---|---|
| Bronze ingestion (Auto Loader) | ✅ Done | Rescue mode, JSON source |
| Silver type conversion | ✅ Done | All 9 tables typed correctly |
| Silver standardization | ✅ Done | Trim, lowercase, state codes, column renames |
| Silver deduplication | ✅ Done | dropDuplicates on PK columns |
| Silver validation | ✅ Done | is_valid flag + rejected DataFrames |
| Silver derived columns | ✅ Done | Duration metrics, date parts, total_item_value |
| Silver write (MERGE) | ✅ Done | Incremental upsert by PK |
| Silver validation queries | ✅ Done | 18 data-quality checks |
| Gold layer (aggregations, joins) | 🔲 Future | Dimensional models, business KPIs |
| Lakeflow SDP conversion | 🔲 Future | When Bronze becomes continuous/streaming |
| Rejected records quarantine tables | 🔲 Future | Write rejected DataFrames to `silver.*_rejected` tables |
| Foreign key enforcement | 🔲 Future | `ALTER TABLE ... ADD CONSTRAINT` on UC tables |